In [2]:
# server.py (updated Flask backend)
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
import os
import time
import uuid
from werkzeug.utils import secure_filename

app = Flask(__name__)
CORS(app)

# Configure upload folder
UPLOAD_FOLDER = 'uploads'
if not os.path.exists(UPLOAD_FOLDER):
    os.makedirs(UPLOAD_FOLDER)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

# In-memory simple storage (you can later replace with a database)
# Structure: { chat_id: [messages] }
chats = {}

@app.route("/send_message", methods=["POST"])
def send_message():
    data = request.json
    text = data.get("text", "")
    chat_id = data.get("chat_id", "default")
    
    if not text.strip():
        return jsonify({"error": "Message is empty"}), 400
    
    # Create chat if it doesn't exist
    if chat_id not in chats:
        chats[chat_id] = []
    
    # Save user message with timestamp
    message = {
        "id": str(uuid.uuid4()),
        "sender": "user",
        "text": text,
        "timestamp": int(time.time() * 1000)  # milliseconds since epoch
    }
    chats[chat_id].append(message)
    
    return jsonify({"status": "Message received", "message": message}), 200

@app.route("/get_messages", methods=["GET"])
def get_messages():
    chat_id = request.args.get("chat_id", "default")
    
    # Return empty list if chat doesn't exist
    if chat_id not in chats:
        return jsonify([])
    
    return jsonify(chats[chat_id])

@app.route("/admin_reply", methods=["POST"])
def admin_reply():
    data = request.json
    text = data.get("text", "")
    chat_id = data.get("chat_id", "default")
    
    if not text.strip():
        return jsonify({"error": "Reply is empty"}), 400
    
    # Create chat if it doesn't exist
    if chat_id not in chats:
        chats[chat_id] = []
    
    # Save admin message with timestamp
    message = {
        "id": str(uuid.uuid4()),
        "sender": "admin",
        "text": text,
        "timestamp": int(time.time() * 1000)  # milliseconds since epoch
    }
    chats[chat_id].append(message)
    
    return jsonify({"status": "Reply sent", "message": message}), 200

@app.route("/get_chats", methods=["GET"])
def get_chats():
    # Return list of chat IDs and their first message as preview
    chat_list = []
    for chat_id, messages in chats.items():
        if messages:
            chat_list.append({
                "id": chat_id,
                "preview": messages[0]["text"][:50] + "..." if len(messages[0]["text"]) > 50 else messages[0]["text"],
                "timestamp": messages[-1]["timestamp"],
                "message_count": len(messages)
            })
    
    return jsonify(chat_list)

@app.route("/upload_file", methods=["POST"])
def upload_file():
    chat_id = request.form.get("chat_id", "default")
    
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400
    
    file = request.files['file']
    
    if file.filename == '':
        return jsonify({"error": "No selected file"}), 400
    
    if file:
        # Create chat if it doesn't exist
        if chat_id not in chats:
            chats[chat_id] = []
        
        # Save file
        filename = secure_filename(file.filename)
        file_path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(file_path)
        
        # Add file message
        message = {
            "id": str(uuid.uuid4()),
            "sender": "user",
            "text": f"[File uploaded: {filename}]",
            "timestamp": int(time.time() * 1000),
            "file": {
                "name": filename,
                "path": file_path
            }
        }
        chats[chat_id].append(message)
        
        return jsonify({
            "status": "File uploaded", 
            "filename": filename,
            "url": f"/files/{filename}",
            "message": message
        }), 200

@app.route("/files/<filename>", methods=["GET"])
def get_file(filename):
    return send_from_directory(app.config['UPLOAD_FOLDER'], filename)

@app.route("/clear_chat", methods=["POST"])
def clear_chat():
    data = request.json
    chat_id = data.get("chat_id", "default")
    
    if chat_id in chats:
        chats[chat_id] = []
    
    return jsonify({"status": "Chat cleared"}), 200

@app.route("/clear_all", methods=["POST"])
def clear_all():
    global chats
    chats = {}
    return jsonify({"status": "All chats cleared"}), 200

if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.108:5000
Press CTRL+C to quit
127.0.0.1 - - [13/May/2025 18:54:28] "GET /get_messages HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "OPTIONS /send_message HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "OPTIONS /send_message HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "GET /get_messages HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "POST /send_message HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "POST /send_message HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "POST /send_message HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "POST /send_message HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "GET /get_messages HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:28] "GET /get_messages HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:30] "GET /get_messages HTTP/1.1" 200 -
127.0.0.1 - - [13/May/2025 18:54:34] "GET /get_messages HTTP/